In [103]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix)
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM,Dense,Dropout,Input)
from tensorflow.keras.callbacks import EarlyStopping

In [104]:
df = pd.read_csv("air_compressor_metro.csv")

In [105]:
df = df.drop(columns = 'Unnamed: 0')

In [106]:
df = df.sort_values("timestamp").reset_index(drop = True)

In [107]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [108]:
df = df.set_index("timestamp")

In [109]:
failure_events = [
    {
        "failure_id": 1,
        "start": pd.Timestamp("2020-04-18 00:00:00"),
        "end": pd.Timestamp("2020-04-18 23:59:00")
    },
    {
        "failure_id": 2,
        "start": pd.Timestamp("2020-05-29 23:30:00"),
        "end": pd.Timestamp("2020-05-30 06:00:00")
    },
    {
        "failure_id": 3,
        "start": pd.Timestamp("2020-06-05 10:00:00"),
        "end": pd.Timestamp("2020-06-07 14:30:00")
    },
    {
        "failure_id": 4,
        "start": pd.Timestamp("2020-07-15 14:30:00"),
        "end": pd.Timestamp("2020-07-15 19:00:00")
    }
]

In [110]:
df["failure"] = 0
DELTA = pd.Timedelta(minutes= 30)
for event in failure_events:
    start_event = event["start"] - DELTA
    mask = ((df.index >= start_event) & (df.index <= event["start"]))
    df.loc[mask, "failure"] = 1

In [111]:
df.value_counts("failure")

failure
0    1516253
1        695
Name: count, dtype: int64

In [112]:
df_10min =  df.resample("10min").agg({    
    "TP2": ["mean", "std", "min", "max"],
    "TP3": ["mean", "std", "min", "max"],
    "H1": ["mean", "std", "min", "max"],
    "DV_pressure": ["mean", "std", "min", "max"],
    "Reservoirs": ["mean", "std", "min", "max"],
    "Oil_temperature": ["mean", "std", "min", "max"],
    "Motor_current": ["mean", "std", "min", "max"],
    "COMP": ["mean", "std"],
    "DV_eletric": ["mean", "std"],
    "Towers": ["mean", "std"],
    "failure": "max"})

In [113]:
df_10min.columns = ["_".join(col).strip("_") for col in df_10min.columns]

In [114]:
df_10min.value_counts("failure_max")

failure_max
0.0    25508
1.0       13
Name: count, dtype: int64

In [115]:
sensor_cols = ["TP2_mean","TP3_mean","H1_mean","DV_pressure_mean","Reservoirs_mean","Oil_temperature_mean","Motor_current_mean"]

In [116]:
for col in sensor_cols:
    df_10min[f"{col}_1h_mean"] = df_10min[col].rolling(6).mean()
    df_10min[f"{col}_1h_std"] = df_10min[col].rolling(6).std()

In [117]:
df_10min.isna().sum().sum()

np.int64(266535)

In [118]:
df_10min = df_10min.copy()

In [119]:
df_10min = df_10min.dropna()

In [120]:
df_10min.isna().sum().sum()

np.int64(0)

In [121]:
df_10min.shape

(24596, 49)

In [122]:
df_10min.value_counts("failure_max")

failure_max
0.0    24583
1.0       13
Name: count, dtype: int64

In [123]:
X = df_10min.drop(columns=["failure_max"])
y = df_10min["failure_max"]

In [ ]:
valid_mask = pd.Series(True, index=X.index)

for event in failure_events:

    start = event["start"]
    end = event["end"]

    valid_mask.loc[(valid_mask.index >= start) & (valid_mask.index <= end)] = False

X = X.loc[valid_mask]
y = y.loc[valid_mask]

In [125]:
folds = []
DELTA = pd.Timedelta(hours=6)
GAP = pd.Timedelta(hours=1)
for event in failure_events[1:]:
    start_event = event["start"] - DELTA 
    end_event = event["end"]
    X_train_mask = X.index <= start_event 
    y_train_mask = y.index <= start_event
    X_test_mask =  (X.index >= start_event) & (X.index < end_event) 
    y_test_mask =  (y.index >= start_event) & (y.index < end_event)
    folds.append({"failure_id": event["failure_id"], "X_train_mask": X_test_mask, "y_train_mask": y_train_mask,
                   "X_test_mask": X_test_mask, "y_test_mask": y_test_mask})
    print(event["failure_id"],
          "train pos:", int(y[y_train_mask].sum()), "| train neg:", int((y[y_train_mask]==0).sum()),
          "| test pos:", int(y[y_test_mask].sum()),  "| test neg:", int((y[y_test_mask]==0).sum()))


2 train pos: 3 | train neg: 13435 | test pos: 3 | test neg: 33
3 train pos: 6 | train neg: 14199 | test pos: 3 | test neg: 33
4 train pos: 9 | train neg: 18511 | test pos: 3 | test neg: 33


In [126]:
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
SEQ_LEN = 12
THRESHOLD = 0.5
X = X.sort_index()
y = y.loc[X.index]
lstm_results = []
lstm_models = {}
lstm_predictions = {}
for fold in folds:
    train_mask = fold["X_train_mask"]
    test_mask = fold["X_test_mask"]

    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
  
    X_test = X.loc[test_mask]
    y_test = y.loc[test_mask]
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_scaled = scaler.transform(X).astype(np.float32)

    X_seq = []
    for i in range(SEQ_LEN - 1, len(X_scaled)):
        sequence = X_scaled[i - SEQ_LEN + 1: i + 1]
        X_seq.append(sequence)
    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = y.iloc[SEQ_LEN - 1:].to_numpy(dtype=np.int32)
    sequence_times = X.index[SEQ_LEN - 1:]

    train_seq_mask = sequence_times.isin(X_train.index)
    test_seq_mask = sequence_times.isin(X_test.index)

    X_train_seq = X_seq[train_seq_mask]
    y_train_seq = y_seq[train_seq_mask]
                
    X_test_seq = X_seq[test_seq_mask]
    y_test_seq = y_seq[test_seq_mask]
    test_times = sequence_times[test_seq_mask]
    
    print("X_seq.shape:", X_seq.shape)
    
    print("y_seq:", y_seq.shape)
    
    print("sequence_times:", sequence_times.shape)
    
    print("Training sequences:", X_train_seq.shape)
                
    print("Test sequences:", X_test_seq.shape)
                
    print("Training class counts:",pd.Series(y_train_seq).value_counts().to_dict())
                
    print("Test class counts:",pd.Series(y_test_seq).value_counts().to_dict())

    n_negative = int((y_train_seq == 0).sum())
    n_positive = int((y_train_seq == 1).sum())
    failure_id = fold["failure_id"]   
    if n_positive == 0 or n_negative == 0:
        raise ValueError(f"Training fold {failure_id} does not contain both classes.")
        
    class_weight = {0: 1.0,1: n_negative / n_positive}
        
    print("Class weights:", class_weight)
    model = Sequential([
            Input(shape=(SEQ_LEN, X.shape[1])),
            LSTM(32),
            Dropout(0.2),
            Dense(16, activation="relu"),
            Dropout(0.2),
            Dense(1, activation="sigmoid")])
    
    model.compile(optimizer="adam",loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.Precision(name="precision"),tf.keras.metrics.Recall(name="recall")])


    early_stopping = EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)
    
    
    history = model.fit(X_train_seq,y_train_seq,validation_split=0.15,epochs=20,batch_size=64,class_weight=class_weight,
                        callbacks=[early_stopping],shuffle=False,verbose=1)

    y_prob = model.predict(X_test_seq,verbose=0).ravel()

    y_pred = (y_prob >= THRESHOLD).astype(int)

    """  print("\nActual labels:")
    print(pd.Series(y_test_seq).value_counts())

    print("\nPredicted labels:")
    print(pd.Series(y_pred).value_counts())

    print("\nProbability statistics:")
    print(pd.Series(y_prob).describe())

    print("\nPositive probabilities:")
    print(y_prob[y_test_seq == 1][:20])

    print("\nNegative probabilities:")
    print(y_prob[y_test_seq == 0][:20]) """

    precision = precision_score(y_test_seq, y_pred, zero_division=0)
    
    recall = recall_score(y_test_seq, y_pred, zero_division=0)
    
    f1 = f1_score(y_test_seq, y_pred, zero_division=0)
    
    roc_auc = roc_auc_score(y_test_seq, y_prob)
    
    pr_auc = average_precision_score(y_test_seq, y_prob)
    
    cm = confusion_matrix(y_test_seq, y_pred, labels=[0, 1])
    
    tn, fp, fn, tp = cm.ravel()
    
    print("\nClassification Report:")
    print(classification_report(y_test_seq,y_pred,labels=[0, 1],target_names=["Healthy", "Failure"],zero_division=0,digits=4) )
    
    print("Confusion Matrix:")
    print(cm)
    
    print(f"Threshold: {THRESHOLD}")
    print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
   

X_seq.shape: (24095, 12, 48)
y_seq: (24095,)
sequence_times: (24095,)
Training sequences: (36, 12, 48)
Test sequences: (36, 12, 48)
Training class counts: {0: 33, 1: 3}
Test class counts: {0: 33, 1: 3}
Class weights: {0: 1.0, 1: 11.0}
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 0.7173 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.7504 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 0.7038 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.7429 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 0.6833 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.7355 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - loss: 0.6737 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 0.7281 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113